In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
import os
import pickle
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                             classification_report, precision_recall_curve,
                             average_precision_score)
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import lightgbm as lgb
import xgboost as xgb
from sklearn.dummy import DummyClassifier

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

# ── Reproducibility ──────────────────────────────────────────────
SEED       = 42
N_FOLDS    = 5
TARGET_COL = 'TARGET'
ID_COL     = 'SK_ID_CURR'

np.random.seed(SEED)
print("✅ All libraries loaded")
print(f"   LightGBM version : {lgb.__version__}")
print(f"   XGBoost version  : {xgb.__version__}")

In [ ]:
print("Loading processed data...")
t = time.time()

train = pd.read_csv(r'C:\Users\Priyanshi Mittal\home-credit-default-risk\data\train_processed.csv')
test  = pd.read_csv(r'C:\Users\Priyanshi Mittal\home-credit-default-risk\data\test_processed.csv')

print(f"Train : {train.shape}  — loaded in {time.time()-t:.1f}s")
print(f"Test  : {test.shape}")
print(f"Target distribution:\n{train[TARGET_COL].value_counts(normalize=True).round(4)}")

# ── Separate features and target ─────────────────────────────────
X = train.drop(columns=[TARGET_COL, ID_COL])
y = train[TARGET_COL]
X_test      = test.drop(columns=[ID_COL], errors='ignore')
test_ids    = test[ID_COL]

# Align columns — test must have exact same columns as train
X_test = X_test.reindex(columns=X.columns, fill_value=0)

print(f"\nX       : {X.shape}")
print(f"y       : {y.shape}  →  default rate: {y.mean()*100:.2f}%")
print(f"X_test  : {X_test.shape}")

In [ ]:
print("=" * 50)
print("CLASS IMBALANCE ANALYSIS")
print("=" * 50)

n_total    = len(y)
n_default  = y.sum()
n_safe     = n_total - n_default
ratio      = n_safe / n_default

print(f"Total samples   : {n_total:,}")
print(f"Non-default (0) : {n_safe:,}  ({n_safe/n_total*100:.1f}%)")
print(f"Default (1)     : {n_default:,}  ({n_default/n_total*100:.1f}%)")
print(f"Imbalance ratio : {ratio:.1f}:1")
print(f"\nscale_pos_weight for LightGBM/XGBoost: {ratio:.2f}")


# scale_pos_weight value to use in LightGBM
SPW = round(ratio, 2)

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

print(f"Cross-validation strategy: {N_FOLDS}-Fold Stratified K-Fold")
print(f"Each fold train size : ~{int(len(X)*0.8):,}")
print(f"Each fold valid size : ~{int(len(X)*0.2):,}")
print()

# Verify stratification
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    fold_pos_rate = y.iloc[val_idx].mean() * 100
    print(f"  Fold {fold+1}: val size={len(val_idx):,}  "
          f"positive rate={fold_pos_rate:.2f}%")

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
import time

print("BASELINE MODELS")
print("=" * 50)

# ── Step 1: Use a SAMPLE for baselines ──────────────────────────
# Baselines are just sanity checks — no need for full 300K rows
# 50K rows gives same reliable estimate in 1/6th the time

SAMPLE_SIZE = 50_000
sample_idx  = np.random.RandomState(SEED).choice(
                  len(X), size=SAMPLE_SIZE, replace=False)

X_sample = X.iloc[sample_idx]
y_sample = y.iloc[sample_idx]

print(f"Using sample: {SAMPLE_SIZE:,} rows (from {len(X):,})")
print(f"Sample default rate: {y_sample.mean()*100:.2f}%  "
      f"(full: {y.mean()*100:.2f}%)  ← should be similar\n")

# ── Step 2: Fill missing ONCE — not inside cross_val_score ───────
# compute median from sample only (fast)
col_medians  = X_sample.median()
X_sample_filled = X_sample.fillna(col_medians)

print(f"Missing values filled with median")
print(f"X_sample_filled shape: {X_sample_filled.shape}\n")

# ── Step 3: Use only 3-fold for baselines (not 5) ────────────────
# Baselines don't need high precision — 3 folds is enough
skf_base = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

# ── Step 4: Use fewer features for Logistic Regression ───────────
# Top 50 features by correlation — logistic regression
# doesn't need all 400, and more features = much slower

top_features = (X_sample_filled.corrwith(y_sample)
                               .abs()
                               .sort_values(ascending=False)
                               .head(50)
                               .index.tolist())

X_lr = X_sample_filled[top_features]
print(f"Logistic Regression using top {len(top_features)} features\n")

# ── Baseline 1: Always predict majority class ────────────────────
t = time.time()
dummy_majority = DummyClassifier(strategy='most_frequent')
scores = []
for tr_idx, val_idx in skf_base.split(X_sample_filled, y_sample):
    dummy_majority.fit(X_sample_filled.iloc[tr_idx],
                       y_sample.iloc[tr_idx])
    pred = dummy_majority.predict_proba(
               X_sample_filled.iloc[val_idx])[:, 1]
    scores.append(roc_auc_score(y_sample.iloc[val_idx], pred))
print(f"Always predict 0 (majority)  →  "
      f"AUC: {np.mean(scores):.4f}  [{time.time()-t:.1f}s]")

# ── Baseline 2: Random predictions ──────────────────────────────
t = time.time()
dummy_random = DummyClassifier(strategy='stratified',
                                random_state=SEED)
scores = []
for tr_idx, val_idx in skf_base.split(X_sample_filled, y_sample):
    dummy_random.fit(X_sample_filled.iloc[tr_idx],
                     y_sample.iloc[tr_idx])
    pred = dummy_random.predict_proba(
               X_sample_filled.iloc[val_idx])[:, 1]
    scores.append(roc_auc_score(y_sample.iloc[val_idx], pred))
print(f"Random predictions           →  "
      f"AUC: {np.mean(scores):.4f}  [{time.time()-t:.1f}s]")

# ── Baseline 3: Logistic Regression (fast version) ───────────────
t = time.time()
lr = LogisticRegression(
    max_iter    = 200,          # reduced from 1000
    C           = 0.01,         # strong regularization = faster convergence
    solver      = 'saga',       # fastest solver for large datasets
    class_weight= 'balanced',
    random_state= SEED,
    n_jobs      = 1             # single job — avoids memory copying
)
scores = []
for tr_idx, val_idx in skf_base.split(X_lr, y_sample):
    lr.fit(X_lr.iloc[tr_idx], y_sample.iloc[tr_idx])
    pred = lr.predict_proba(X_lr.iloc[val_idx])[:, 1]
    scores.append(roc_auc_score(y_sample.iloc[val_idx], pred))
print(f"Logistic Regression          →  "
      f"AUC: {np.mean(scores):.4f}  [{time.time()-t:.1f}s]")

# ── Baseline 4: EXT_SOURCE only (manual baseline) ────────────────
# Since EXT_SOURCE is the best single predictor,
# this is a very strong manual baseline — no model needed
t = time.time()
ext_cols_available = [c for c in ['EXT_SOURCE_1','EXT_SOURCE_2',
                                    'EXT_SOURCE_3','EXT_SOURCE_MEAN']
                       if c in X.columns]
ext_score = X[ext_cols_available].mean(axis=1).fillna(0)

# Negate because higher EXT_SOURCE = lower default
ext_auc = roc_auc_score(y, 1 - ext_score)
print(f"EXT_SOURCE mean (no model)   →  "
      f"AUC: {ext_auc:.4f}  [{time.time()-t:.1f}s]")
print("  ↑ This alone beats most complex models!")

# ── Summary ──────────────────────────────────────────────────────
print(f"""
{'='*50}
BASELINE SUMMARY
{'='*50}
  Random guess       : ~0.5000  (floor)
  Always predict 0   : ~0.5000  (useless)
  Logistic Regression: ~0.68-0.71
  EXT_SOURCE only    : ~0.74-0.76  (strong!)
  ─────────────────────────────────
  LightGBM target    : ~0.78-0.80  (what we aim for)
  Competition top    :  0.8057     (ceiling)
{'='*50}

Your LightGBM must beat {ext_auc:.4f} to be worth using.
""")

In [ ]:
lgb_params = {
    # Core
    'objective'        : 'binary',        # binary classification
    'metric'           : 'auc',           # evaluate with ROC-AUC
    'boosting_type'    : 'gbdt',          # gradient boosted decision trees

    # Tree structure
    'num_leaves'       : 31,              # max leaves per tree (complexity control)
    'max_depth'        : -1,              # -1 = no limit (num_leaves controls it)
    'min_child_samples': 20,              # min samples in leaf (prevents overfitting)
    'min_child_weight' : 1e-3,

    # Sampling (regularization)
    'feature_fraction' : 0.8,            # use 80% features per tree
    'bagging_fraction' : 0.8,            # use 80% samples per tree
    'bagging_freq'     : 5,              # apply bagging every 5 iterations

    # Learning
    'learning_rate'    : 0.05,           # small lr + more trees = better
    'n_estimators'     : 2000,           # max trees (early stopping controls actual)

    # Imbalance
    'scale_pos_weight' : SPW,            # handle 12:1 class imbalance

    # Regularization
    'reg_alpha'        : 0.1,            # L1 regularization
    'reg_lambda'       : 0.1,            # L2 regularization
    'min_split_gain'   : 0.01,

    # Speed & reproducibility
    'n_jobs'           : -1,
    'random_state'     : SEED,
    'verbose'          : -1,             # suppress output
}

print("LightGBM Parameters:")
for k, v in lgb_params.items():
    print(f"  {k:<25}: {v}")

In [ ]:
# ── Storage for results ──────────────────────────────────────────
oof_preds   = np.zeros(len(X))          # out-of-fold predictions
test_preds  = np.zeros(len(X_test))     # test predictions
fold_scores = []
feature_importances = pd.DataFrame()

print("Training LightGBM with 3-Fold Cross Validation...")
print("=" * 60)

total_start = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    fold_start = time.time()
    print(f"\n{'─'*50}")
    print(f"FOLD {fold+1}/{N_FOLDS}")
    print(f"{'─'*50}")

    # ── Split data ───────────────────────────────────────────────
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    print(f"  Train: {X_tr.shape}  |  Val: {X_val.shape}")
    print(f"  Train positive rate: {y_tr.mean()*100:.2f}%")
    print(f"  Val   positive rate: {y_val.mean()*100:.2f}%")

    # ── Train model ──────────────────────────────────────────────
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set              = [(X_val, y_val)],
        eval_metric           = 'auc',
        callbacks             = [
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=200)    # print every 200 trees
        ]
    )

    # ── Predict ──────────────────────────────────────────────────
    val_pred  = model.predict_proba(X_val)[:, 1]
    test_pred = model.predict_proba(X_test)[:, 1]

    # ── Store OOF predictions ────────────────────────────────────
    oof_preds[val_idx] = val_pred
    test_preds         += test_pred / N_FOLDS   # average across folds

    # ── Score ────────────────────────────────────────────────────
    fold_auc = roc_auc_score(y_val, val_pred)
    fold_scores.append(fold_auc)

    # ── Feature importance ───────────────────────────────────────
    fold_importance = pd.DataFrame({
        'feature'    : X.columns,
        'importance' : model.feature_importances_,
        'fold'       : fold + 1
    })
    feature_importances = pd.concat([feature_importances, fold_importance])

    fold_time = time.time() - fold_start
    print(f"\n  ✅ Fold {fold+1} ROC-AUC : {fold_auc:.5f}")
    print(f"     Best iteration   : {model.best_iteration_}")
    print(f"     Time             : {fold_time:.1f}s")

    # Save each fold model
    os.makedirs('../models', exist_ok=True)
    with open(f'../models/lgb_fold{fold+1}.pkl', 'wb') as f:
        pickle.dump(model, f)

# ── Overall Results ──────────────────────────────────────────────
total_time = time.time() - total_start
oof_auc    = roc_auc_score(y, oof_preds)

print(f"\n{'='*60}")
print(f"LIGHTGBM CROSS-VALIDATION RESULTS")
print(f"{'='*60}")
for i, score in enumerate(fold_scores):
    print(f"  Fold {i+1}: {score:.5f}")
print(f"{'─'*40}")
print(f"  Mean  : {np.mean(fold_scores):.5f}")
print(f"  Std   : {np.std(fold_scores):.5f}")
print(f"  OOF   : {oof_auc:.5f}  ← most reliable score")
print(f"{'─'*40}")
print(f"  Total training time: {total_time/60:.1f} minutes")

In [ ]:
import psutil

ram   = psutil.virtual_memory()
print(f"Total RAM    : {ram.total/1e9:.1f} GB")
print(f"Available    : {ram.available/1e9:.1f} GB")
print(f"Used         : {ram.percent:.1f}%")

# Dataset memory usage
print(f"\nX memory     : {X.memory_usage(deep=True).sum()/1e9:.2f} GB")
print(f"X_test memory: {X_test.memory_usage(deep=True).sum()/1e9:.2f} GB")

if ram.available < 4e9:
    print("\n⚠️  WARNING: Less than 4GB RAM available!")
    print("   Close Chrome, VS Code extensions, other apps")
    print("   Or reduce X to float32:")
    print("   X = X.astype('float32')")

In [ ]:
# Reduce memory by half — float32 instead of float64
X      = X.astype('float32')
X_test = X_test.astype('float32')

print(f"X memory after float32: {X.memory_usage(deep=True).sum()/1e9:.2f} GB")
# Should drop from ~1GB to ~0.5GB

In [ ]:
# Use only top 200 features by correlation
top_200 = (X.corrwith(y)
            .abs()
            .sort_values(ascending=False)
            .head(200)
            .index.tolist())

X_fast      = X[top_200]
X_test_fast = X_test[top_200]

print(f"Reduced: {X.shape[1]} → {X_fast.shape[1]} features")
# Training time drops by ~50%

In [ ]:
lgb_params_fast = {
    'objective'        : 'binary',
    'metric'           : 'auc',
    'boosting_type'    : 'gbdt',

    'num_leaves'       : 31,
    'learning_rate'    : 0.05,
    'n_estimators'     : 1000,    # reduced from 2000

    'feature_fraction' : 0.7,     # use 70% features (was 80%)
    'bagging_fraction' : 0.7,     # use 70% samples (was 80%)
    'bagging_freq'     : 5,

    'scale_pos_weight' : SPW,
    'reg_alpha'        : 0.1,
    'reg_lambda'       : 0.1,

    'n_jobs'           : -1,
    'random_state'     : SEED,
    'verbose'          : -1,
}
# ~40% faster than original params

In [ ]:
skf = StratifiedKFold(n_splits=3,   # was 5
                       shuffle=True,
                       random_state=SEED)
# 40% less time
# AUC estimate difference: ~0.001 (negligible)

In [ ]:
print("""
WHY WE USE OOF (Out-Of-Fold) PREDICTIONS:
─────────────────────────────────────────
Simple CV:  train → validate → get AUC → AVERAGE AUCs
OOF:        each sample gets predicted EXACTLY ONCE when it was in validation

OOF advantages:
  1. Single unified AUC over entire dataset (more stable)
  2. Can plot ROC curve for full training data
  3. Can use OOF predictions as features in stacking (advanced)
  4. Better calibrated probability estimates

Our OOF AUC vs fold mean:
""")
print(f"  Fold mean AUC : {np.mean(fold_scores):.5f}")
print(f"  OOF AUC       : {oof_auc:.5f}")
print(f"  Difference    : {abs(np.mean(fold_scores) - oof_auc):.5f}  (should be tiny)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Plot 1: ROC Curve ────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(y, oof_preds)
axes[0].plot(fpr, tpr, color='#e74c3c', linewidth=2,
              label=f'LightGBM OOF (AUC = {oof_auc:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
axes[0].set_xlabel('False Positive Rate (FPR)', fontsize=12)
axes[0].set_ylabel('True Positive Rate (TPR)', fontsize=12)
axes[0].set_title('ROC Curve — OOF Predictions', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# ── Plot 2: Precision-Recall Curve ───────────────────────────────
# More informative than ROC for imbalanced datasets
precision, recall, pr_thresh = precision_recall_curve(y, oof_preds)
ap_score = average_precision_score(y, oof_preds)

axes[1].plot(recall, precision, color='#3498db', linewidth=2,
              label=f'LightGBM (AP = {ap_score:.4f})')
axes[1].axhline(y.mean(), color='gray', linestyle='--',
                 label=f'Random baseline ({y.mean():.3f})')
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision-Recall Curve\n(Better for Imbalanced Data)', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/model_01_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"ROC-AUC  : {oof_auc:.4f}")
print(f"Avg Prec : {ap_score:.4f}")

In [ ]:
print("THRESHOLD OPTIMIZATION")
print("=" * 50)
print("""
Default threshold = 0.5 (predict default if prob > 0.5)
But with 8% positive rate, optimal threshold is usually MUCH lower.
We choose threshold based on business objective:
  - Minimize false negatives → catch as many defaulters as possible
  - Balance precision/recall based on cost of each error
""")

# Calculate metrics at different thresholds
thresholds_to_try = np.arange(0.05, 0.55, 0.01)
results = []

for thresh in thresholds_to_try:
    preds_binary = (oof_preds >= thresh).astype(int)
    tn = ((preds_binary == 0) & (y == 0)).sum()
    fp = ((preds_binary == 1) & (y == 0)).sum()
    fn = ((preds_binary == 0) & (y == 1)).sum()
    tp = ((preds_binary == 1) & (y == 1)).sum()

    precision_ = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall_    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_        = (2 * precision_ * recall_ / (precision_ + recall_)
                  if (precision_ + recall_) > 0 else 0)

    results.append({
        'threshold' : thresh,
        'precision' : precision_,
        'recall'    : recall_,
        'f1'        : f1_,
        'tp'        : tp, 'fp': fp, 'tn': tn, 'fn': fn
    })

thresh_df = pd.DataFrame(results)
best_f1_row = thresh_df.loc[thresh_df['f1'].idxmax()]
BEST_THRESHOLD = best_f1_row['threshold']

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(thresh_df['threshold'], thresh_df['precision'],
         label='Precision', color='#3498db', linewidth=2)
ax.plot(thresh_df['threshold'], thresh_df['recall'],
         label='Recall', color='#e74c3c', linewidth=2)
ax.plot(thresh_df['threshold'], thresh_df['f1'],
         label='F1 Score', color='#2ecc71', linewidth=2)
ax.axvline(BEST_THRESHOLD, color='orange', linestyle='--', linewidth=1.5,
            label=f'Best threshold = {BEST_THRESHOLD:.2f}')
ax.set_xlabel('Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Precision / Recall / F1 vs Threshold', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/model_02_threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nBest threshold (max F1): {BEST_THRESHOLD:.2f}")
print(f"At this threshold:")
print(f"  Precision : {best_f1_row['precision']:.3f}  "
      f"(of predicted defaulters, {best_f1_row['precision']*100:.1f}% actually default)")
print(f"  Recall    : {best_f1_row['recall']:.3f}  "
      f"(caught {best_f1_row['recall']*100:.1f}% of all actual defaulters)")
print(f"  F1 Score  : {best_f1_row['f1']:.3f}")
print(f"  TP={int(best_f1_row['tp']):,}  FP={int(best_f1_row['fp']):,}  "
      f"TN={int(best_f1_row['tn']):,}  FN={int(best_f1_row['fn']):,}")

In [ ]:
final_preds_binary = (oof_preds >= BEST_THRESHOLD).astype(int)
cm = confusion_matrix(y, final_preds_binary)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt=',', cmap='Blues', ax=ax,
            xticklabels=['Predicted: Repay', 'Predicted: Default'],
            yticklabels=['Actual: Repay', 'Actual: Default'])

ax.set_title(f'Confusion Matrix (threshold={BEST_THRESHOLD:.2f})', fontsize=13)

# Annotate with business meaning
total = cm.sum()
ax.set_xlabel(f'Total predictions: {total:,}', fontsize=11)

plt.tight_layout()
plt.savefig('../outputs/model_03_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nClassification Report:")
print(classification_report(y, final_preds_binary,
                             target_names=['Repay (0)', 'Default (1)']))

print("""
Business Interpretation:
  True Positive  (TP) → Correctly identified defaulters  ✅ (caught bad loans)
  True Negative  (TN) → Correctly identified repayers    ✅ (approved good loans)
  False Positive (FP) → Good customer flagged as default ❌ (lost business)
  False Negative (FN) → Defaulter missed by model        ❌ (financial loss)

In credit risk: FN is more costly than FP
→ Missing a defaulter = financial loss
→ Rejecting a good customer = lost revenue only
""")


In [ ]:
# Average importance across all folds
mean_importance = (feature_importances
                   .groupby('feature')['importance']
                   .mean()
                   .sort_values(ascending=False)
                   .reset_index())

print(f"Total features: {len(mean_importance)}")
print(f"\nTop 20 most important features:")
print(mean_importance.head(20).to_string(index=False))

# ── Plot top 40 ──────────────────────────────────────────────────
top_n = 40
top_feats = mean_importance.head(top_n)

fig, ax = plt.subplots(figsize=(12, 14))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, top_n))[::-1]
bars = ax.barh(top_feats['feature'][::-1],
                top_feats['importance'][::-1],
                color=colors)

ax.set_xlabel('Average Feature Importance (split count)', fontsize=12)
ax.set_title(f'Top {top_n} Features — LightGBM (averaged over {N_FOLDS} folds)',
              fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/model_04_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Feature source breakdown ─────────────────────────────────────
print("\nFeature Importance by Source Table:")
prefixes = {
    'EXT_SOURCE'  : 'Application (EXT_SOURCE)',
    'BURO_'       : 'Bureau',
    'PREV_'       : 'Previous Application',
    'POS_'        : 'POS Cash',
    'CC_'         : 'Credit Card',
    'INS_'        : 'Installments',
}
for prefix, label in prefixes.items():
    mask  = mean_importance['feature'].str.startswith(prefix)
    total = mean_importance.loc[mask, 'importance'].sum()
    count = mask.sum()
    pct   = total / mean_importance['importance'].sum() * 100
    print(f"  {label:<30}: {count:3d} features  {pct:5.1f}% importance")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── Plot 1: Score distribution by class ──────────────────────────
axes[0].hist(oof_preds[y==0], bins=80, alpha=0.6,
              color='#2ecc71', label='Actual: Repay (0)', density=True)
axes[0].hist(oof_preds[y==1], bins=80, alpha=0.6,
              color='#e74c3c', label='Actual: Default (1)', density=True)
axes[0].axvline(BEST_THRESHOLD, color='orange', linestyle='--',
                 linewidth=2, label=f'Threshold={BEST_THRESHOLD:.2f}')
axes[0].set_xlabel('Predicted Probability of Default', fontsize=11)
axes[0].set_ylabel('Density', fontsize=11)
axes[0].set_title('Score Distribution by True Class', fontsize=12)
axes[0].legend()

# ── Plot 2: Calibration curve ────────────────────────────────────
from sklearn.calibration import calibration_curve
fraction_pos, mean_pred = calibration_curve(y, oof_preds, n_bins=20)

axes[1].plot(mean_pred, fraction_pos, 's-',
              color='#e74c3c', linewidth=2, label='LightGBM')
axes[1].plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
axes[1].set_xlabel('Mean Predicted Probability', fontsize=11)
axes[1].set_ylabel('Fraction of Positives', fontsize=11)
axes[1].set_title('Calibration Curve\n(How well do probabilities reflect reality?)',
                   fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/model_05_score_dist_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
Calibration interpretation:
  Points on diagonal   → perfectly calibrated (prob 0.3 = 30% actually default)
  Points above diagonal → underestimates risk (model too optimistic)
  Points below diagonal → overestimates risk (model too pessimistic)
""")

In [ ]:
print("MULTI-MODEL COMPARISON")
print("=" * 60)

model_results = {'LightGBM': {'oof': oof_preds, 'auc': oof_auc}}

# ── XGBoost ──────────────────────────────────────────────────────
xgb_params = {
    'tree_method'      : 'hist',
    'objective'        : 'binary:logistic',
    'eval_metric'      : 'auc',
    'max_depth'        : 5,
    'learning_rate'    : 0.05,
    'n_estimators'     : 1000,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'scale_pos_weight' : SPW,
    'reg_alpha'        : 0.1,
    'reg_lambda'       : 0.1,
    'random_state'     : SEED,
    'n_jobs'           : -1,
    'verbosity'        : 0,
}

xgb_oof   = np.zeros(len(X))
xgb_test  = np.zeros(len(X_test))
xgb_scores = []

print("\nTraining XGBoost...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx].fillna(0), X.iloc[val_idx].fillna(0)
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    xgb_model = xgb.XGBClassifier(early_stopping_rounds=50, **xgb_params)
    xgb_model.fit(X_tr, y_tr,
              eval_set=[(X_val, y_val)],
              verbose=False)

    val_pred          = xgb_model.predict_proba(X_val)[:, 1]
    xgb_oof[val_idx]  = val_pred
    xgb_test          += xgb_model.predict_proba(X_test.fillna(0))[:, 1] / N_FOLDS
    xgb_scores.append(roc_auc_score(y_val, val_pred))
    print(f"  Fold {fold+1}: {xgb_scores[-1]:.5f}")

xgb_auc = roc_auc_score(y, xgb_oof)
model_results['XGBoost'] = {'oof': xgb_oof, 'auc': xgb_auc}
print(f"XGBoost OOF AUC: {xgb_auc:.5f}")

# ── Results Summary ──────────────────────────────────────────────
print(f"\n{'='*50}")
print(f"MODEL COMPARISON SUMMARY")
print(f"{'='*50}")
for name, res in model_results.items():
    print(f"  {name:<15} OOF AUC: {res['auc']:.5f}")

In [ ]:
print("ENSEMBLE: Blending LightGBM + XGBoost")
print("=" * 50)

# Simple weighted average
weights = {
    'lgb' : 0.6,    # LightGBM gets more weight if AUC is higher
    'xgb' : 0.4,
}

# OOF ensemble
ensemble_oof  = (weights['lgb'] * oof_preds +
                  weights['xgb'] * xgb_oof)
ensemble_auc  = roc_auc_score(y, ensemble_oof)

# Test ensemble
ensemble_test = (weights['lgb'] * test_preds +
                  weights['xgb'] * xgb_test)

print(f"  LightGBM OOF AUC   : {oof_auc:.5f}  (weight: {weights['lgb']})")
print(f"  XGBoost  OOF AUC   : {xgb_auc:.5f}  (weight: {weights['xgb']})")
print(f"  Ensemble OOF AUC   : {ensemble_auc:.5f}")
print(f"\n  Improvement over best single: "
      f"{ensemble_auc - max(oof_auc, xgb_auc):.5f}")

# ROC comparison plot
fig, ax = plt.subplots(figsize=(10, 7))
for name, res, color in zip(
    ['LightGBM', 'XGBoost', 'Ensemble'],
    [oof_preds, xgb_oof, ensemble_oof],
    ['#e74c3c', '#3498db', '#2ecc71']
):
    fpr, tpr, _ = roc_curve(y, res)
    auc = roc_auc_score(y, res)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc:.4f})')

ax.plot([0,1],[0,1],'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison: LightGBM vs XGBoost vs Ensemble')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/model_06_model_comparison_roc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("Generating submission files...")
os.makedirs('../outputs', exist_ok=True)

# ── LightGBM submission ──────────────────────────────────────────
lgb_submission = pd.DataFrame({
    'SK_ID_CURR' : test_ids.values,
    'TARGET'     : test_preds
})
lgb_submission.to_csv('../outputs/submission_lgb.csv', index=False)

# ── XGBoost submission ───────────────────────────────────────────
xgb_submission = pd.DataFrame({
    'SK_ID_CURR' : test_ids.values,
    'TARGET'     : xgb_test
})
xgb_submission.to_csv('../outputs/submission_xgb.csv', index=False)

# ── Ensemble submission ──────────────────────────────────────────
ens_submission = pd.DataFrame({
    'SK_ID_CURR' : test_ids.values,
    'TARGET'     : ensemble_test
})
ens_submission.to_csv('../outputs/submission_ensemble.csv', index=False)

# ── Validate format ──────────────────────────────────────────────
print("\nSubmission file validation:")
for name, df in [('LightGBM', lgb_submission),
                  ('XGBoost',  xgb_submission),
                  ('Ensemble', ens_submission)]:
    print(f"\n  {name}:")
    print(f"    Shape   : {df.shape}")
    print(f"    Columns : {df.columns.tolist()}")
    print(f"    Score range: [{df['TARGET'].min():.4f}, {df['TARGET'].max():.4f}]")
    print(f"    No nulls: {df.isnull().sum().sum() == 0}")
    print(df.head(3))

In [ ]:
# ── Save OOF predictions for stacking later ──────────────────────
oof_df = pd.DataFrame({
    'SK_ID_CURR'   : train[ID_COL].values,
    'TARGET'       : y.values,
    'LGB_OOF'      : oof_preds,
    'XGB_OOF'      : xgb_oof,
    'ENSEMBLE_OOF' : ensemble_oof,
})
oof_df.to_csv('../outputs/oof_predictions.csv', index=False)

# ── Final summary ────────────────────────────────────────────────
print("=" * 60)
print("FINAL MODELING SUMMARY")
print("=" * 60)
print(f"""
Dataset
  Train rows  : {len(X):,}
  Features    : {X.shape[1]:,}
  Default rate: {y.mean()*100:.2f}%

Cross Validation (5-Fold Stratified)
  LightGBM OOF AUC  : {oof_auc:.5f}
  XGBoost  OOF AUC  : {xgb_auc:.5f}
  Ensemble OOF AUC  : {ensemble_auc:.5f}

Threshold Analysis
  Best threshold    : {BEST_THRESHOLD:.2f}

Files saved
  ../outputs/submission_lgb.csv
  ../outputs/submission_xgb.csv
  ../outputs/submission_ensemble.csv
  ../outputs/oof_predictions.csv
  ../models/lgb_fold1-5.pkl
""")